In [ ]:
from __future__ import annotations

from pathlib import Path
import os

import numpy as np
import pandas as pd
import pydicom
import torch
from skimage.transform import resize
from torch.utils.data import DataLoader, Dataset


def ct_normalization(img, meta):
    """
    Normaliza el rango HU a [0, 1].
    Ajustamos de -1024 (aire) a 3072 (tejido denso) para cubrir un rango amplio
    para resaltar detalles anatómicos.
    """
    def scale(img, meta, slope=1.0, intersept=-1024):

        slope = float(meta.get("Rescale Slope", slope))
        intersept = float(meta.get("Rescale Intercept", intersept))
        
        img = img * slope + intersept  # Aplica la transformación de escala y desplazamiento
        img = resize(img, (256, 256), preserve_range=True)  # Asegura que todas las imágenes tengan el mismo tamaño
        img = (img + 1024) / (3072 + 1024)  # Escala a [0, 1]
        return img.astype(np.float32)
    
    return scale(img, meta)


def _read_dicom(path: Path) -> np.ndarray:
    
    ds = pydicom.dcmread(path)  # Lee el archivo DICOM y convierte a numpy array
    metadata = {}
    for element in ds:
            if getattr(element, "name", None) == "Pixel Data":
                continue
            try:
                name = str(element.name)
                value = element.value
                if value is None:
                    continue
                metadata[name] = str(value)
            except Exception:
                continue

    return ct_normalization(ds.pixel_array, metadata)


class DatasetLoader(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        dim: int = 2,
        training: str = "model",
    ) -> None:
        """
        Args:
            df: DataFrame with either:
                - dim == 2: columns ['Case', 'SDCT_file', 'LDCT_file'] or ['Case', 'SDCT', 'LDCT']
                - dim == 3: columns ['Case', 'SDCT', 'LDCT'] where SDCT/LDCT are volume directories
            transform: Optional transform to be applied on a sample.
            training: "model" or "vae", used to determine how to process the data.
        """
        self.dim = dim
        self.df = df # = self._build_index(df, dim)
        self.training = training

        self._preprocess()
        self._tensor_loader()

    
    def _preprocess(self):
        self.df = self.load_slices()

        if self.training == "model":
            if self.dim == 3:
                self.df = self.df.groupby('Case').agg({
                    'SDCT': list,
                    'LDCT': list
                }).reset_index()
            self.df = self.df.dropna(subset=['SDCT', 'LDCT'])
        elif self.training == "vae":
            if self.dim == 3:
                self.df = self.df.groupby('Case').agg({
                    'SDCT': list,
                }).reset_index()
            self.df = self.df.dropna(subset=['SDCT'])

    def _tensor_loader(self):
        if self.training == "model":
            modalities = ["SDCT", "LDCT"]
        elif self.training == "vae":
            modalities = ["SDCT"]
        for idx in range(len(self.df)):
            self.load_data(idx, to_load=modalities)

    def load_slices(self) -> pd.DataFrame:
        rows = []
        for row in self.df.itertuples(index=False):
            sdct_dir = Path(row.SDCT)
            sdct_files = sorted(sdct_dir.glob("*.dcm"))

            if self.training == "model":
                ldct_files = sorted(Path(row.LDCT).glob("*.dcm"))
                rows.extend([
                    {'SDCT': s, 'LDCT': l, 'Case': row.Case} 
                    for s, l in zip(sdct_files, ldct_files)
                ])
            elif self.training == "vae":
                rows.extend([{'SDCT': s, 'Case': row.Case} for s in sdct_files])
                
        return pd.DataFrame(rows)
    
    def load_data(self, idx: int, to_load: list[str] = ["SDCT", "LDCT"]) -> dict:
        row = self.df.iloc[idx]
        if self.dim == 3:
            for modality in to_load:
                volume = np.array([_read_dicom(f) for f in sorted(row[modality])]) # Shape: (D, H, W)
                tensor = torch.from_numpy(volume).unsqueeze(0).float()  # Shape: (1, D, H, W)
                self.df.iloc[idx][modality] = tensor  # Cache the loaded tensor in the DataFrame
        else:
            for modality in to_load:
                img = _read_dicom(Path(row[modality]))
                tensor = torch.from_numpy(img).unsqueeze(0).float()  # Shape: (1, H, W)
                self.df.iloc[idx][modality] = tensor

    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        sample = {'Case': row['Case']}
        if self.training == "model":
            sample['SDCT'] = row['SDCT']
            sample['LDCT'] = row['LDCT']
        elif self.training == "vae":
            sample['SDCT'] = row['SDCT']
        
        return sample

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train = False
dataset = pd.read_csv(os.path.join("Dataset", 'train.txt' if train else 'test.txt'), sep='\t', names=('Case','SDCT','LDCT',))
print("Dataset loaded successfully.")
dataset = DatasetLoader(dataset, dim=3, training="vae")
print("Dataset object created successfully.")

Dataset loaded successfully.


/tmp/ipykernel_2441478/4133447112.py:123: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.df.iloc[idx][modality] = tensor  # Cache the loaded tensor in the DataFrame


In [ ]:
loader = DataLoader(dataset, batch_size=1, shuffle=True)
loader = iter(loader)  # Convertimos el DataLoader a un iterador para usar next()

In [ ]:
batch = next(loader)

In [ ]:
print(batch)

In [ ]:
import torch.nn.functional as F

batch = next(loader)  # Obtenemos el primer batch
torch.cuda.empty_cache()
inputs = batch['SDCT'].to(device)  # Shape: (B, D, H, W)

divisor = 16
actual_d = inputs.shape[2]

# Calculamos cuánto falta para el siguiente múltiplo
pad_needed = (divisor - (actual_d % divisor)) % divisor

if pad_needed > 0:
    inputs = F.pad(inputs, (0, 0, 0, 0, 0, pad_needed), value=0)
    print(f"Se agregaron {pad_needed} cortes. Nueva profundidad: {inputs.shape[2]}")

In [ ]:
# Dimensiones deseadas del cubo
cube_size = 16

# Calculamos los puntos de inicio para centrar el recorte
# inputs shape: [B, C, D, H, W] -> [1, 1, 280, 256, 256]
d_start = (inputs.shape[2] - cube_size) // 2
h_start = (inputs.shape[3] - cube_size) // 2
w_start = (inputs.shape[4] - cube_size) // 2

# Extraemos el cubo usando slicing de PyTorch
perfect_cube = inputs[:, :, 
                      d_start : d_start + cube_size, 
                      h_start : h_start + cube_size, 
                      w_start : w_start + cube_size]

print(f"Forma del cubo: {perfect_cube.shape}") 
# Resultado: torch.Size([1, 1, 128, 128, 128])

In [ ]:
import torch
from medvae import MVAE
import matplotlib.pyplot as plt

# --- 1. Configuración del dispositivo ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs!")
#     model = nn.DataParallel(model)

# --- 2. Carga del Modelo ---
try:
    model = MVAE(model_name="medvae_4_1_3d", modality="ct").to(device)
    model.eval()
    print("¡Modelo MVAE cargado correctamente!")
except Exception as e:
    print(f"Error al cargar MVAE: {e}")

with torch.no_grad():
    # 1. Obtener el latente
    reconstruction, latent = model(inputs, decode=True)
    print(f"Dimensiones latentes:  {latent.shape} (Reducción 4x)")
    print(f"Dimensiones reconstrucción: {reconstruction.shape}")

In [ ]:
# Si agregaste 8 cortes al final, se los quitas así:
reconstruction = reconstruction[:actual_d, :, :]
# --- 5. Visualización ---
plt.figure(figsize=(15, 5))

# 1. Entrada (Suponiendo que inputs es [Slices, H, W] o [1, 1, Slices, H, W])
# Si inputs es [1, 1, 200, 256, 256], lo limpiamos con .squeeze()
input_clean = inputs.squeeze()

plt.subplot(1, 3, 1)
plt.title(f"Entrada (Slice {2})")
plt.imshow(input_clean[1].cpu().detach(), cmap="gray")
plt.axis("off")

# 2. Espacio Latente (Shape: [50, 64, 64])
# Tomamos la slice central del espacio comprimido
plt.subplot(1, 3, 2)
plt.title(f"Latente (Slice {2})")
plt.imshow(latent[1].cpu().detach(), cmap="magma")
plt.axis("off")

# 3. Reconstrucción (Shape: [200, 256, 256])
recon_clean = reconstruction.squeeze()

plt.subplot(1, 3, 3)
plt.title(f"Reconstrucción (Slice {2})")
plt.imshow(recon_clean[1].cpu().detach(), cmap="gray")
plt.axis("off")

plt.tight_layout()
plt.show()

# Prints de verificación
print(f"Forma final procesada:")
print(f"Entrada: {input_clean.shape}")
print(f"Latente: {latent.shape}")
print(f"Reconstrucción: {recon_clean.shape}")
torch.cuda.empty_cache()

In [ ]:
# 2. Espacio Latente (Shape: [50, 64, 64])
# Se plotea cada slice del espacio latente como un subplot en forma de matriz

plt.figure(figsize=(20, 20))

for i in range(latent.shape[0]//2):
    plt.subplot(10, 5, i + 1)  # Ajusta el número de filas y columnas según el número de slices
    plt.title(f"Latente Slice {i}")
    plt.imshow(latent[i].cpu().detach(), cmap="magma")
    plt.axis("off")

In [ ]:
# 2. Espacio Latente (Shape: [50, 64, 64])
# Se plotea cada slice del espacio latente como un subplot en forma de matriz

plt.figure(figsize=(20, 20))

for i in range(33):
    plt.subplot(10, 5, i + 1)  # Ajusta el número de filas y columnas según el número de slices
    plt.title(f"Reconstrucción Slice {i + 1}")
    plt.imshow(recon_clean[i].cpu().detach(), cmap="gray")
    plt.axis("off")